# HaloRT - Parameter Sweep

A second example: sweeping scene parameters (anisotropy `g` and light
color) with HaloRT, to see how the model responds to material/lighting
changes at interactive speed. Assumes `01_getting_started.ipynb` has
already been run once in this environment (or re-run its install +
setup cells here).

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q rendervous
    %pip install -q git+https://github.com/rendervous/HaloRT.git
    print("Installed rendervous + halort (pretrained weights are bundled - no repo clone needed)")
else:
    print("Not running in Colab - make sure rendervous and halort are already installed in this environment")


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import rdv
import halort

rdv.device(0)


## Scene setup (same synthetic cloud/sky as the getting-started notebook)

In [ ]:
def make_synthetic_cloud(size=96, seed=0):
    rng = np.random.default_rng(seed)
    d, h, w = size, size, size
    zz, yy, xx = np.mgrid[0:d, 0:h, 0:w].astype(np.float32)
    center = np.array([d, h, w]) / 2
    zz, yy, xx = zz - center[0], yy - center[1], xx - center[2]
    radius = size * 0.32
    dist = np.sqrt(xx**2 + (yy * 1.3)**2 + zz**2)
    base = np.clip(1.0 - dist / radius, 0.0, 1.0) ** 1.5
    noise = np.zeros((d, h, w), dtype=np.float32)
    amp = 1.0
    for octave_size in (8, 16, 32):
        coarse = rng.random((octave_size, octave_size, octave_size)).astype(np.float32)
        upsampled = torch.nn.functional.interpolate(
            torch.from_numpy(coarse)[None, None], size=(d, h, w), mode="trilinear", align_corners=False
        )[0, 0].numpy()
        noise += amp * (upsampled - 0.5)
        amp *= 0.5
    density = np.clip(base + 0.6 * noise * base, 0.0, None)
    return torch.from_numpy(density[..., None]).float()


def make_synthetic_sky(h=128, w=256):
    yy = np.linspace(1.0, 0.0, h, dtype=np.float32)[:, None]
    horizon = np.array([0.9, 0.85, 0.75], dtype=np.float32)
    zenith = np.array([0.25, 0.45, 0.85], dtype=np.float32)
    sky = horizon[None, None, :] * (1 - yy)[..., None] + zenith[None, None, :] * yy[..., None]
    return torch.from_numpy(np.broadcast_to(sky, (h, w, 3)).copy()).float()


cloud = make_synthetic_cloud().cuda()
environment = make_synthetic_sky().cuda()

halort_net = halort.create_scatnet("halort", width=512, height=512).cuda()

halort_net.set_extinction(cloud, density_scale=300.0)
halort_net.set_scattering_albedo((0.99, 0.99, 0.99))
halort_net.set_environment(environment)
halort_net.set_camera(0, origin=(0.0, 0.5, 2.6), target=(0.0, 0.0, 0.0), up=(0.0, 1.0, 0.0))
halort_net.set_light(light_direction=(0.3, 0.8, 0.4), light_intensity=(3.0, 2.9, 2.7))


def tonemap(img):
    a = img[0].detach().float().cpu().numpy()
    return np.clip(np.clip(a, 0.0, None) ** (1.0 / 2.2), 0.0, 1.0)


## Sweep anisotropy `g`

From strongly back-scattering (`g < 0`) to strongly forward-scattering (`g` near 1).

In [ ]:
g_values = [-0.6, -0.2, 0.0, 0.3, 0.6, 0.85]

fig, axes = plt.subplots(1, len(g_values), figsize=(3 * len(g_values), 3))
for ax, g in zip(axes, g_values):
    halort_net.set_anisotropy(g)
    with torch.no_grad():
        img = halort_net.reconstruct_radiance_fused()
    ax.imshow(np.flipud(tonemap(img)))
    ax.set_title(f"g = {g}")
    ax.axis("off")
plt.tight_layout()
plt.show()

halort_net.set_anisotropy(0.3)  # reset


## Sweep light color

Fixed intensity magnitude, varying RGB balance.

In [ ]:
light_colors = {
    "white": (1.0, 1.0, 1.0),
    "warm": (1.0, 0.7, 0.4),
    "cool": (0.5, 0.7, 1.0),
    "red": (1.0, 0.3, 0.2),
    "green": (0.3, 1.0, 0.4),
}

fig, axes = plt.subplots(1, len(light_colors), figsize=(3 * len(light_colors), 3))
for ax, (name, color) in zip(axes, light_colors.items()):
    intensity = tuple(3.0 * c for c in color)
    halort_net.set_light(light_direction=(0.3, 0.8, 0.4), light_intensity=intensity)
    with torch.no_grad():
        img = halort_net.reconstruct_radiance_fused()
    ax.imshow(np.flipud(tonemap(img)))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Timing

HaloRT is designed for real-time use - a quick FPS check.

In [ ]:
import time

halort_net.set_light(light_direction=(0.3, 0.8, 0.4), light_intensity=(3.0, 2.9, 2.7))

with torch.no_grad():
    for _ in range(3):  # warm-up
        halort_net.reconstruct_radiance_fused()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    reps = 30
    for _ in range(reps):
        halort_net.reconstruct_radiance_fused()
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0

print(f"{reps / dt:.1f} FPS ({1000 * dt / reps:.2f} ms/frame) at 512x512")
